#Imports

In [145]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
from scipy import interpolate
from scipy.signal import find_peaks
import warnings
from sklearn.model_selection import GridSearchCV
#warnings.filterwarnings('ignore')

 # Classes and Methods Definition

In [146]:
class ICGSubtypeClassifier:
    """
    Impedance Cardiography Complex Subtype Classifier using Pattern Recognition
    Artificial Neural Networks (PRANN) with divide-and-conquer approach.

    Based on: Benouar et al. (2021) "Classification of impedance cardiography
    dZ/dt complex subtypes using pattern recognition artificial neural networks"
    """

    def __init__(self, target_freq=257, window_size_sec=1):
        self.target_freq = target_freq
        self.window_size_sec = window_size_sec
        self.window_size_samples = target_freq * window_size_sec

        # Neural networks (now using manual CV)
        self.autoencoder = None
        self.prann1 = None
        self.subprann1 = None
        self.subprann2 = None

        # Scalers
        self.scaler = MinMaxScaler()

        # Training history
        self.training_history = {}

    def build_autoencoder(self, input_dim, hidden_neurons=10):
        """
        Build autoencoder for synthetic data generation.
        Two feedforward networks with logarithmic-sigmoid activation.
        """
        # Encoder
        encoder_input = layers.Input(shape=(input_dim,))
        encoded = layers.Dense(hidden_neurons, activation='relu', name='encoder_hidden')(encoder_input)

        # Noise
        noisy_input = layers.GaussianNoise(0.1)(encoder_input)

        # Decoder
        decoded = layers.Dense(input_dim, activation='sigmoid', name='decoder_output')(encoded)

        # Autoencoder model
        autoencoder = keras.Model(encoder_input, decoded, name='autoencoder')

        # Compile with scaled conjugate gradient equivalent (Adam optimizer)
        autoencoder.compile(optimizer='adam', loss='mse', metrics=['mse'])

        return autoencoder

    def generate_synthetic_data(self, x1z_segments, labels, balance_classes=True):
        """
        Generate synthetic data using autoencoder to balance classes.
        Train separate autoencoder for each subtype.
        """
        unique_labels = np.unique(labels)
        synthetic_segments = []
        synthetic_labels = []

        if balance_classes:
            # Find the maximum class count for balancing
            max_count = max([np.sum(labels == label) for label in unique_labels])

        for label in unique_labels:
            class_indices = np.where(labels == label)[0]
            class_segments = x1z_segments[class_indices]

            if len(class_segments) == 0:
                continue

            # Build and train autoencoder for this class
            autoencoder = self.build_autoencoder(class_segments.shape[1])

            print(f"Label {label}")

            # Train autoencoder
            #autoencoder.fit(class_segments, class_segments,
            #              epochs=100, batch_size=16, verbose=0,
            #              validation_split=0.2)

            # Generate synthetic data
            if balance_classes:
                n_synthetic = max_count - len(class_segments)

                if n_synthetic > 0:
                    # Generate synthetic samples


                    for r in range(0, n_synthetic, n_synthetic % len(class_segments)):
                        synthetic_samples = autoencoder.predict(class_segments[:n_synthetic % len(class_segments)])

                        synthetic_segments.extend(synthetic_samples)
                        synthetic_labels.extend([label] * len(synthetic_samples))

                    print(len(synthetic_segments), len(synthetic_labels))

            # Add original samples
            print("**********")
            synthetic_segments.extend(class_segments)
            synthetic_labels.extend([label] * len(class_segments))

        return np.array(synthetic_segments), np.array(synthetic_labels)

    def build_prann(self, input_dim, n_classes, hidden_neurons=32, alpha=0.01, dropout_rate=0.3):
        """
        Build Pattern Recognition Artificial Neural Network (PRANN).
        Two-layer feedforward network with sigmoid and softmax layers.
        """
        model = keras.Sequential([
            layers.Input(shape=(input_dim,)),

            # First hidden layer
            layers.Dense(hidden_neurons, activation='relu', name='hidden_layer_1'),
            layers.LeakyReLU(alpha),
            layers.BatchNormalization(),
            layers.Dropout(dropout_rate),

            # Optional: Second hidden layer (if needed)
            layers.Dense(hidden_neurons // 2, activation='relu', name='hidden_layer_2'),
            layers.LeakyReLU(alpha),
            layers.BatchNormalization(),
            layers.Dropout(dropout_rate),

            # Output layer
            layers.Dense(n_classes, activation='softmax', name='output_layer')
        ])

        # Compile with scaled conjugate gradient equivalent
        model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )

        return model

    def create_divide_conquer_targets(self, labels):
        """
        Create targets for divide-and-conquer approach:
        PRANN1: Groups subtypes based on YOZ wave similarity
        - P1C1: ABEXYOZ subtypes 0, 1
        - P1C2: ABEXYOZ subtypes 2, 3, 4
        - P1C3: ABEXYOZ subtype u (undefined) - discarded
        """
        prann1_targets = np.zeros_like(labels)

        for i, label in enumerate(labels):
            if label in [0, 1]:
                prann1_targets[i] = 0  # P1C1
            elif label in [2, 3, 4]:
                prann1_targets[i] = 1  # P1C2
            else:  # undefined or other
                prann1_targets[i] = 2  # P1C3 (will be discarded)

        return prann1_targets

    def train_prann_system(self, x1z_segments, labels, test_size=0.3, validation_size=0.15, hidden_neurons=32, alpha=0.01, dropout_rate=0.3, synthetic = True):
        """
        Train the complete PRANN system using divide-and-conquer approach with manual CV.
        """
        print("Starting PRANN system training with manual cross-validation...")


        print("Generating synthetic data...")
        # For now, using original data (uncomment synthetic generation if needed)
        if synthetic:
          synthetic_segments, synthetic_labels = self.generate_synthetic_data(x1z_segments, labels, balance_classes=True)
        else:
          synthetic_segments, synthetic_labels = x1z_segments, labels
        # Split data: 70% training, 15% validation, 15% testing

        X_temp, X_test, y_temp, y_test = train_test_split(
            synthetic_segments, synthetic_labels, test_size=test_size,
            stratify=synthetic_labels, random_state=42)

        val_size_adjusted = validation_size / (1 - test_size)
        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp, test_size=val_size_adjusted,
            stratify=y_temp, random_state=42)

        # Create PRANN1 targets (divide-and-conquer first level)
        y_train_p1 = self.create_divide_conquer_targets(y_train)
        y_val_p1 = self.create_divide_conquer_targets(y_val)
        y_test_p1 = self.create_divide_conquer_targets(y_test)

        # Remove undefined class (P1C3) for PRANN1 training
        valid_mask_train = y_train_p1 != 2
        valid_mask_val = y_val_p1 != 2
        valid_mask_test = y_test_p1 != 2

        X_train_p1 = X_train[valid_mask_train]
        y_train_p1 = y_train_p1[valid_mask_train]
        X_val_p1 = X_val[valid_mask_val]
        y_val_p1 = y_val_p1[valid_mask_val]

        # Train PRANN1 with manual cross-validation
        print("Training PRANN1 with manual cross-validation...")

        early_stopping = keras.callbacks.EarlyStopping(
                    monitor='val_accuracy', patience=20, restore_best_weights=True)

        self.prann1 = self.build_prann(X_train_p1.shape[1], 2, hidden_neurons=hidden_neurons ,dropout_rate=dropout_rate, alpha=alpha)

        history_prann1 = self.prann1.fit(
            X_train_p1, y_train_p1,
            validation_data=(X_val_p1, y_val_p1),
            epochs=1000, batch_size=32, verbose=0,
            callbacks=[early_stopping]
        )

        self.training_history['PRANN1'] = history_prann1
        # Predict PRANN1 outputs for SubPRANN training
        train_pred_p1 = self.prann1.predict(X_train_p1, verbose=0)
        train_pred_p1_class = np.argmax(train_pred_p1, axis=1)

        # Train SubPRANN1 (for P1C1: subtypes 0, 1)
        mask_p1c1_train = (y_train_p1 == 0)
        if np.sum(mask_p1c1_train) > 0:
            print("Training SubPRANN1...")
            X_train_sub1 = X_train_p1[mask_p1c1_train]
            y_train_sub1 = y_train[valid_mask_train][mask_p1c1_train]

            # Only include subtypes 0 and 1
            valid_sub1 = np.isin(y_train_sub1, [0, 1])
            X_train_sub1 = X_train_sub1[valid_sub1]
            y_train_sub1 = y_train_sub1[valid_sub1]

            if len(np.unique(y_train_sub1)) > 1:
                self.subprann1 = self.build_prann(X_train_sub1.shape[1], len(np.unique(y_train_sub1)), hidden_neurons=hidden_neurons ,dropout_rate=dropout_rate, alpha=alpha)

                # Create validation set for SubPRANN1
                X_train_sub1_split, X_val_sub1, y_train_sub1_split, y_val_sub1 = train_test_split(
                    X_train_sub1, y_train_sub1, test_size=0.2, stratify=y_train_sub1, random_state=42)

                early_stopping = keras.callbacks.EarlyStopping(
                    monitor='val_accuracy', patience=30, restore_best_weights=True)

                history_sub1 = self.subprann1.fit(
                    X_train_sub1_split, y_train_sub1_split,
                    validation_data=(X_val_sub1, y_val_sub1),
                    epochs=1000, batch_size=32, verbose=0,
                    callbacks=[early_stopping]
                )
                self.training_history['SubPRANN1'] = history_sub1

        # Train SubPRANN2 (for P1C2: subtypes 2, 3, 4)
        mask_p1c2_train = (y_train_p1 == 1)
        if np.sum(mask_p1c2_train) > 0:
            print("Training SubPRANN2...")
            X_train_sub2 = X_train_p1[mask_p1c2_train]
            y_train_sub2 = y_train[valid_mask_train][mask_p1c2_train]

            # Only include subtypes 2, 3, and 4
            valid_sub2 = np.isin(y_train_sub2, [2, 3, 4])
            X_train_sub2 = X_train_sub2[valid_sub2]
            y_train_sub2 = y_train_sub2[valid_sub2] - 2

            if len(np.unique(y_train_sub2)) > 1:
                self.subprann2 = self.build_prann(X_train_sub2.shape[1], len(np.unique(y_train_sub2)), hidden_neurons=hidden_neurons ,dropout_rate=dropout_rate, alpha=alpha)

                # Create validation set for SubPRANN2
                X_train_sub2_split, X_val_sub2, y_train_sub2_split, y_val_sub2 = train_test_split(
                    X_train_sub2, y_train_sub2, test_size=0.2, stratify=y_train_sub2, random_state=42)

                early_stopping = keras.callbacks.EarlyStopping(
                    monitor='val_accuracy', patience=30, restore_best_weights=True)

                history_sub2 = self.subprann2.fit(
                    X_train_sub2_split, y_train_sub2_split,
                    validation_data=(X_val_sub2, y_val_sub2),
                    epochs=1000, batch_size=32, verbose=0,
                    callbacks=[early_stopping]
                )
                self.training_history['SubPRANN2'] = history_sub2

        # Evaluate system on test set
        print("Evaluating PRANN system...")
        test_predictions = self.predict(X_test)
        test_accuracy = accuracy_score(y_test, test_predictions)

        print(f"Overall test accuracy: {test_accuracy:.3f}")

        return {
            'test_accuracy': test_accuracy,
            'test_data': (X_test, y_test),
            'predictions': test_predictions
        }

    def predict(self, X):
        """
        Make predictions using the complete PRANN system.
        """
        if self.prann1 is None:
            raise ValueError("PRANN system not trained. Call train_prann_system first.")

        # Step 1: PRANN1 classification
        prann1_pred = self.prann1.predict(X, verbose=0)
        prann1_classes = np.argmax(prann1_pred, axis=1)

        final_predictions = np.full(len(X), -1)  # Initialize with invalid class

        # Step 2: SubPRANN1 for P1C1 samples
        if self.subprann1 is not None:
            p1c1_mask = prann1_classes == 0
            if np.sum(p1c1_mask) > 0:
                sub1_pred = self.subprann1.predict(X[p1c1_mask], verbose=0)
                final_predictions[p1c1_mask] = np.argmax(sub1_pred, axis=1)

        # Step 3: SubPRANN2 for P1C2 samples
        if self.subprann2 is not None:
            p1c2_mask = prann1_classes == 1
            if np.sum(p1c2_mask) > 0:
                sub2_pred = self.subprann2.predict(X[p1c2_mask], verbose=0)
                sub2_classes = np.argmax(sub2_pred, axis=1)
                # Map back to original class labels (2, 3, 4)
                final_predictions[p1c2_mask] = sub2_classes + 2

        return final_predictions

    def evaluate_performance(self, y_true, y_pred):
        """
        Calculate comprehensive performance metrics.
        """
        accuracy = accuracy_score(y_true, y_pred)
        precision, recall, f1, support = precision_recall_fscore_support(
            y_true, y_pred, average='weighted', zero_division=0)

        # Per-class metrics
        precision_per_class, recall_per_class, f1_per_class, _ = precision_recall_fscore_support(
            y_true, y_pred, average=None, zero_division=0)

        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'precision_per_class': precision_per_class,
            'recall_per_class': recall_per_class,
            'f1_per_class': f1_per_class,
            'confusion_matrix': confusion_matrix(y_true, y_pred)
        }

    def plot_training_history(self):
        """
        Plot training history for all networks.
        """
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))

        networks = ['PRANN1', 'SubPRANN1', 'SubPRANN2']

        for i, network in enumerate(networks):
            if network in self.training_history:
                history = self.training_history[network]

                # Accuracy plot
                axes[0, i].plot(history.history['accuracy'], label='Training')
                axes[0, i].plot(history.history['val_accuracy'], label='Validation')
                axes[0, i].set_title(f'{network} Accuracy')
                axes[0, i].set_xlabel('Epoch')
                axes[0, i].set_ylabel('Accuracy')
                axes[0, i].legend()
                axes[0, i].grid(True)

                # Loss plot
                axes[1, i].plot(history.history['loss'], label='Training')
                axes[1, i].plot(history.history['val_loss'], label='Validation')
                axes[1, i].set_title(f'{network} Loss')
                axes[1, i].set_xlabel('Epoch')
                axes[1, i].set_ylabel('Loss')
                axes[1, i].legend()
                axes[1, i].grid(True)
            else:
                axes[0, i].text(0.5, 0.5, f'{network}\nNot Trained',
                               ha='center', va='center', transform=axes[0, i].transAxes)
                axes[1, i].text(0.5, 0.5, f'{network}\nNot Trained',
                               ha='center', va='center', transform=axes[1, i].transAxes)

        plt.tight_layout()
        plt.show()


# Main

In [147]:
from re import A
classifier = ICGSubtypeClassifier()

data = pd.read_csv('heartCycle_full.csv').iloc[:, :]

train = data.iloc[:,:]


all_x1z_segments = np.array(train.iloc[:,2:-1])
all_labels = np.array(train.iloc[:,-1])

print(f"Total processed segments: {len(all_x1z_segments)}")
print(f"Segment shape: {all_x1z_segments.shape}")
print(f"Label distribution: {np.bincount(all_labels)}")

# Train the PRANN system
print("\nTraining PRANN system...")

### UNCOMMENT FOR CROSS-VALIDATION
parameters = {
    "hidden_neurons": [2**1,2**2,2**3,2**4,2**5,2**6,2**7,2**8,2**9,2**10],
    "dropout_rate": [0.1, 0.2, 0.3, 0.4, 0.5],
    "alpha": [0.01, 0.02, 0.03, 0.04, 0.05]
}

metrics = {
    "hidden_neurons": [],
    "dropout_rate": [],
    "alpha": [],
    "accuracy": []
}

for n in parameters["hidden_neurons"]:
  for d in parameters["dropout_rate"]:
    for a in parameters["alpha"]:

      print(f"Trying configuration: \n hidden_neurons={n} \n dropout_rate={d} \n alpha={a}")
      cv = classifier.train_prann_system(all_x1z_segments, all_labels, test_size=0.1, validation_size=0.1, hidden_neurons=n ,dropout_rate=d, alpha=a, synthetic=True)

      metrics["hidden_neurons"].append(n)
      metrics["dropout_rate"].append(d)
      metrics["alpha"].append(a)
      metrics["accuracy"].append(cv['test_accuracy'])

pd.DataFrame(metrics).to_csv("cross_validation_results.csv")

# Evaluate performance

result = classifier.train_prann_system(all_x1z_segments, all_labels, test_size=0.1, validation_size=0.1, hidden_neurons=128, dropout_rate=0.1, alpha=0.01, synthetic=True) #Add the best configuration according to csv file

performance = classifier.evaluate_performance(result['test_data'][1], result['predictions'])

print(f"\nFinal Results:")
print(f"Accuracy: {performance['accuracy']:.3f}")
print(f"Precision: {performance['precision']:.3f}")
print(f"Recall: {performance['recall']:.3f}")
print(f"F1-Score: {performance['f1_score']:.3f}")
print(f"\nConfusion Matrix:")
print(performance['confusion_matrix'])

# Plot training history
#classifier.plot_training_history()

Total processed segments: 1570
Segment shape: (1570, 116)
Label distribution: [272 534 136 309 319]

Training PRANN system...
Trying configuration: 
 hidden_neurons=2 
 dropout_rate=0.1 
 alpha=0.01
Starting PRANN system training with manual cross-validation...
Generating synthetic data...
Label 0
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
262 262
**********
Label 1
**********
Label 2
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
1572 1572
**********
Label 3
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
1933 1933
**********
Label 4
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step 
2457 2457
**********
Training PRANN1 with manual cross-validation...
Training SubPRANN1...
Training SubPRANN2...
Evaluating PRANN system...
Overall test accuracy: 0.514
Trying configuration: 
 hidden_neurons=2 
 dropout_rate=0.1 
 alpha=0.02
Starting PRANN system training with manual cross-validation...
Generating synthetic data.